# Gold Platform Performance

## Import and load Dataset

In [32]:
import pandas as pd

from ecommerce_ingestion.config.constants import DB_OUTPUT_GOLD


- number of games de juegos
- average price
- mean price
- user score average
- meta score average
mixed score mean
- stock percentile
- diferent genders
platform ranking by score/price

In [33]:
gold_game_catalog = pd.read_parquet(DB_OUTPUT_GOLD / "gold_game_catalog.parquet")
gold_game_catalog["category_name"].unique()

<ArrowStringArray>
[         'nintendo-64',                  'wii',             'gamecube',
               'switch',        'xbox-platform',            'dreamcast',
        'playstation-3', 'playstation-platform',        'playstation-2',
                   'pc',     'game-boy-advance',        'playstation-4',
                  '3ds',             'xbox-360',                'wii-u',
     'playstation-vita',                   'ds',             'xbox-one',
                  'psp',        'playstation-5',        'xbox-series-x',
               'stadia']
Length: 22, dtype: str

## Group Table

In [34]:

gold_platform_performance = gold_game_catalog.groupby("category_name").agg(
    game_count=("game_id", "count"),
    average_user_score=("user_score", "mean"),
    average_meta_score=("meta_score", "mean"),
    average_price=("current_price", "mean"),
    median_price=("current_price", "median"),
    different_primary_genres=("primary_genre", "nunique"),
    different_secondary_genres=("secondary_genre", "nunique"),
    different_tertiary_genres=("tertiary_genre", "nunique"),
    in_stock_percentage=("stock_status", lambda x: (x.eq("in_stock").mean()) * 100),
).reset_index()
gold_platform_performance["average_mixed_score"] = gold_platform_performance[
    ["average_user_score", "average_meta_score"]].mean(axis=1)
gold_platform_performance["score_price_value"] = gold_platform_performance["average_mixed_score"] / gold_platform_performance["average_price"]

In [35]:
gold_platform_performance["category_name"].shape

(22,)

In [40]:
gold_platform_performance.sort_values("score_price_value", ascending=False).head(10)

,category_name,game_count,average_user_score,average_meta_score,average_price,median_price,different_primary_genres,different_secondary_genres,different_tertiary_genres,in_stock_percentage,average_mixed_score,score_price_value
19,xbox-one,97,63.092784,75.278351,64.082784,65.99,10,12,5,45.360825,69.185567,1.079628
10,playstation-5,13,69.307692,75.769231,69.143846,71.99,4,5,1,61.538462,72.538462,1.049095
9,playstation-4,261,67.758621,75.639847,68.702644,70.99,12,15,5,51.724138,71.699234,1.043617
21,xbox-series-x,4,70.000000,77.750000,70.990000,75.99,4,3,1,50.000000,73.875000,1.040640
14,stadia,2,74.500000,80.500000,75.490000,75.49,1,1,1,100.000000,77.500000,1.026626
6,pc,961,72.810614,77.264308,73.754828,75.99,12,16,9,49.635796,75.037461,1.017390
18,xbox-360,187,73.962567,76.882353,74.952567,76.99,11,14,7,49.197861,75.422460,1.006269
0,3ds,91,74.505495,77.065934,75.495495,77.99,9,13,6,45.054945,75.785714,1.003844
8,playstation-3,129,75.038760,77.418605,75.990000,77.99,10,11,8,48.837209,76.228682,1.003141
15,switch,256,75.292969,76.960938,76.271250,77.99,12,13,6,49.218750,76.126953,0.998108


## Analysis NaNs

In [37]:
gold_platform_performance.isna().sum()

category_name                 0
game_count                    0
average_user_score            0
average_meta_score            0
average_price                 0
median_price                  0
different_primary_genres      0
different_secondary_genres    0
different_tertiary_genres     0
in_stock_percentage           0
average_mixed_score           0
score_price_value             0
dtype: int64

## Save Table

In [38]:
gold_platform_performance.to_parquet(DB_OUTPUT_GOLD / "gold_platform_performance.parquet")